In [ ]:
# 1. Install required packages
!pip install -q -U transformers trl peft bitsandbytes datasets accelerate


In [ ]:
# 2. TRAINING HYPERPARAMETERS (Adjust here if needed)
MODEL_ID = "Qwen/Qwen3-8B"  # Base model name on Hugging Face
MAX_LENGTH = 768            # Maximum sequence length (optimized for Kaggle GPU memory)
BATCH_SIZE = 2              # Training batch size per device (can be increased to 4 on L4/A100)
GRADIENT_ACCUMULATION = 4   # Gradient accumulation steps (Effective batch size = BATCH_SIZE * GRADIENT_ACCUMULATION = 8)
EPOCHS = 2                  # Number of training epochs
LEARNING_RATE = 2e-4        # Learning rate
OUTPUT_DIR = "/kaggle/working/results"  # Output directory on Kaggle


In [ ]:
# 3. Search and load NL -> FOL translation datasets
import os
import glob
import json

# Common Kaggle directories to search for dataset files
POSSIBLE_PATHS = [
    ".",
    "/kaggle/working",
    "/kaggle/input",
    "/kaggle/input/exact-dataset",
]

def find_file(filename):
    for path in POSSIBLE_PATHS:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            return full_path
    for path in POSSIBLE_PATHS:
        matches = glob.glob(os.path.join(path, "**", filename), recursive=True)
        if matches:
            return matches[0]
    return None

logic_path = find_file("logic_based.json")
folio_path = find_file("folio-train.json")

if not logic_path or not folio_path:
    raise FileNotFoundError(
        "Could not find logic_based.json or folio-train.json.\n"
        "Please upload them to /kaggle/working/ or add them as a Kaggle Dataset."
    )

print(f"Using logic_based path: {logic_path}")
print(f"Using folio-train path: {folio_path}")

def load_translation_dataset(l_path, f_path):
    samples = []
    seen_premises = set()
    
    with open(l_path, "r", encoding="utf-8") as f:
        logic_data = json.load(f)
    for item in logic_data:
        nl_list = item.get("premises-NL", [])
        fol_list = item.get("premises-FOL", [])
        if not nl_list or not fol_list or len(nl_list) != len(fol_list):
            continue
        nl_serialized = "\n".join(nl_list)
        if nl_serialized in seen_premises:
            continue
        seen_premises.add(nl_serialized)
        samples.append({"premises-NL": nl_list, "premises-FOL": fol_list})
    print(f"Loaded {len(samples)} unique samples from logic_based.json")
    
    logic_count = len(samples)
    with open(f_path, "r", encoding="utf-8") as f:
        folio_data = json.load(f)
    for item in folio_data:
        nl_list = item.get("premises-NL", [])
        fol_list = item.get("premises-FOL", [])
        if not nl_list or not fol_list or len(nl_list) != len(fol_list):
            continue
        nl_serialized = "\n".join(nl_list)
        if nl_serialized in seen_premises:
            continue
        seen_premises.add(nl_serialized)
        samples.append({"premises-NL": nl_list, "premises-FOL": fol_list})
    print(f"Loaded {len(samples) - logic_count} unique samples from folio-train.json")
    print(f"Total unique samples after deduplication: {len(samples)}")
    return samples

raw_samples = load_translation_dataset(logic_path, folio_path)


In [ ]:
# 4. Initialize Tokenizer and load Base Model in 4-bit QLoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute (Optimal for L4/A100).")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute (Standard for T4/P100).")

# Select the most optimal Attention implementation
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("FlashAttention-2 is installed. Using flash_attention_2.")
except ImportError:
    print("FlashAttention-2 not found. Using PyTorch Native SDPA (Scaled Dot Product Attention).")

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    attn_implementation=attn_impl
)
base_model.config.use_cache = False

if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Resume from previous checkpoints if adapter_config.json exists in OUTPUT_DIR
adapter_config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
if os.path.exists(adapter_config_path):
    print(f"Resuming PEFT adapter weights from {OUTPUT_DIR}...")
    model = PeftModel.from_pretrained(base_model, OUTPUT_DIR, is_trainable=True)
else:
    print("Initializing a new PEFT adapter...")
    model = get_peft_model(base_model, peft_config)

model.print_trainable_parameters()


In [ ]:
# 5. Format Dataset (Chat Template) and split Train/Val
import os
import json
from datasets import Dataset

# Define prompt templates for flat JSON list output
system_prompt_template = (
    "You convert natural-language premises into parser-safe first-order logic formulas.\n\n"
    "Output a STRICT valid JSON list of strings containing the first-order logic formulas in the exact order of the input premises.\n\n"
    "ALLOWED OPERATORS:\n"
    "AND, OR, NOT, ->, <->, =, !=, >=, <=, >, <, ForAll, Exists\n\n"
    "QUANTIFIER RULES:\n"
    "Use nested quantifiers only. E.g., ForAll(x, ForAll(y, P(x,y)))\n\n"
    "Return JSON only."
)

user_prompt_template = (
    "Convert the following premises into canonical first-order logic.\n\n"
    "Premises:\n"
    "{{premises}}\n\n"
    "Return a JSON list of strings containing the formulas."
)

formatted_samples = []
for item in raw_samples:
    nl_list = item["premises-NL"]
    fol_list = item["premises-FOL"]
    
    # Format premises as a numbered list
    nl_content = ""
    for i, nl in enumerate(nl_list, start=1):
        nl_content += f"{i}. {nl}\n"
        
    # Render user prompt template
    user_prompt = user_prompt_template.replace("{{premises}}", nl_content.strip())
    
    # Assistant response is a flat JSON list of FOL formulas
    assistant_response = json.dumps(fol_list, indent=2)
    
    formatted_samples.append({
        "messages": [
            {"role": "system", "content": system_prompt_template},
            {"role": "user", "content": user_prompt.strip()},
            {"role": "assistant", "content": assistant_response.strip()}
        ]
    })

dataset = Dataset.from_list(formatted_samples)

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True, remove_columns=["messages"])
dataset = dataset.shuffle(seed=42)
split_dataset = dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")


In [ ]:
# 6. Configure SFTTrainer and start training
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import glob

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    eval_strategy="epoch",   # Evaluate at the end of each epoch to maximize training speed
    save_strategy="epoch",   # Save checkpoints at the end of each epoch
    learning_rate=LEARNING_RATE,
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=1,
    logging_first_step=True,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    gradient_checkpointing=True if torch.cuda.is_available() else False,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_eval_batch_size=BATCH_SIZE,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    dataset_text_field="text",
    max_length=MAX_LENGTH
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=sft_config,
    callbacks=[LossLoggingCallback()]
)

# Automatically resume from the latest checkpoint if it exists
checkpoints = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
resume_path = None
if checkpoints:
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    resume_path = checkpoints[-1]
    print(f"Found checkpoints. Resuming training from: {resume_path}")

trainer.train(resume_from_checkpoint=resume_path)

# Save best model to results directory
print(f"Saving best validation adapter weights and tokenizer to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training completed successfully!")


In [ ]:
# 7. Running Inference Test
import gc
import json

# Clean up GPU memory for stable inference
if "trainer" in globals():
    del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.eval()

test_nl_premises = [
    "All humans are mortal.",
    "Socrates is a human."
]

test_nl_content = ""
for i, nl in enumerate(test_nl_premises, start=1):
    test_nl_content += f"{i}. {nl}\n"

test_user_prompt = user_prompt_template.replace("{{premises}}", test_nl_content.strip())

messages = [
    {"role": "system", "content": system_prompt_template},
    {"role": "user", "content": test_user_prompt.strip()}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, eos_token_id=tokenizer.eos_token_id)

response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- NL Premises ---")
print(test_nl_content.strip())
print("\n--- Model Generated FOL Translation (Expected JSON List) ---")
print(response_text)
print("--------------------------------------------------")
